<h1>SQLAlchemy Tutorial<h1/>

In [6]:
import sqlalchemy

In [7]:
sqlalchemy.__version__

'1.4.49'

In [8]:
from sqlalchemy import create_engine, text

In [9]:
engine = create_engine("sqlite+pysqlite:///:memory:", echo=True)

In [10]:
with engine.connect() as conn:
    result =  conn.execute(text("select 'hello world'"))
    print(result.all())

2026-01-24 00:05:03,679 INFO sqlalchemy.engine.Engine select 'hello world'
2026-01-24 00:05:03,682 INFO sqlalchemy.engine.Engine [generated in 0.00259s] ()
[('hello world',)]


In [11]:
# Commit as you go"
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE some_table (x int, y int)"))
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"), 
        [{"x": 1, "y": 1}, {"x": 2, "y": 4}],
    )
    conn.commit()

2026-01-24 00:05:03,710 INFO sqlalchemy.engine.Engine CREATE TABLE some_table (x int, y int)
2026-01-24 00:05:03,712 INFO sqlalchemy.engine.Engine [generated in 0.00178s] ()
2026-01-24 00:05:03,717 INFO sqlalchemy.engine.Engine COMMIT
2026-01-24 00:05:03,719 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-24 00:05:03,720 INFO sqlalchemy.engine.Engine [generated in 0.00102s] ((1, 1), (2, 4))
2026-01-24 00:05:03,721 INFO sqlalchemy.engine.Engine COMMIT


C:\Users\Goldstine\AppData\Local\Temp\ipykernel_22948\3838845129.py:3: RemovedIn20Warning: Deprecated API features detected! These feature(s) are not compatible with SQLAlchemy 2.0. To prevent incompatible upgrades prior to updating applications, ensure requirements files are pinned to "sqlalchemy<2.0". Set environment variable SQLALCHEMY_WARN_20=1 to show all deprecation warnings.  Set environment variable SQLALCHEMY_SILENCE_UBER_WARNING=1 to silence this message. (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  conn.execute(text("CREATE TABLE some_table (x int, y int)"))


AttributeError: 'Connection' object has no attribute 'commit'

In [ ]:
# begins once#
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 6, "y": 8}, {"x": 9, "y": 10}],
    )

2026-01-23 23:43:24,953 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:24,954 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-23 23:43:24,955 INFO sqlalchemy.engine.Engine [cached since 0.02979s ago] [(6, 8), (9, 10)]
2026-01-23 23:43:24,958 INFO sqlalchemy.engine.Engine COMMIT


In [ ]:
# begins once#
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (9, 10)")
            )

2026-01-23 23:43:24,970 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:24,972 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (9, 10)
2026-01-23 23:43:24,973 INFO sqlalchemy.engine.Engine [generated in 0.00138s] ()
2026-01-23 23:43:24,975 INFO sqlalchemy.engine.Engine COMMIT


In [ ]:
# select statement#
with engine.begin() as conn:
    query_result = conn.execute(text("SELECT * FROM some_table"))
    print(query_result.all())

2026-01-23 23:43:25,000 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:25,001 INFO sqlalchemy.engine.Engine SELECT * FROM some_table
2026-01-23 23:43:25,003 INFO sqlalchemy.engine.Engine [generated in 0.00135s] ()
[(1, 1), (2, 4), (6, 8), (9, 10), (9, 10)]
2026-01-23 23:43:25,005 INFO sqlalchemy.engine.Engine COMMIT


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table"))
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-23 23:43:25,032 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:25,033 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table
2026-01-23 23:43:25,034 INFO sqlalchemy.engine.Engine [generated in 0.00236s] ()
x: 1 y: 1
x: 2 y: 4
x: 6 y: 8
x: 9 y: 10
x: 9 y: 10
2026-01-23 23:43:25,036 INFO sqlalchemy.engine.Engine ROLLBACK


<h2>Sending Parameters<h2/>

In [ ]:
# return value of y where its value is greater than a specific value)

with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table WHERE y > :y"), {"y": 8})
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-23 23:43:25,729 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:25,730 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ?
2026-01-23 23:43:25,731 INFO sqlalchemy.engine.Engine [generated in 0.00214s] (8,)
x: 9 y: 10
x: 9 y: 10
2026-01-23 23:43:25,733 INFO sqlalchemy.engine.Engine ROLLBACK


<h2>Sending Multiple Parameters<h2/>

In [ ]:
# inserting multiple records in a sql statement

with engine.connect() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"), 
        [{"x": 11, "y": 12}, {"x": 13, "y": 14}]
    )
    conn.commit()

2026-01-23 23:43:26,299 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-23 23:43:26,300 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-23 23:43:26,301 INFO sqlalchemy.engine.Engine [cached since 1.375s ago] [(11, 12), (13, 14)]
2026-01-23 23:43:26,302 INFO sqlalchemy.engine.Engine COMMIT


<h2>Executing with an ORM Session<h2/>

In [13]:
from sqlalchemy.orm import Session

In [14]:
stmt = text("SELECT x, y FROM some_table WHERE y > :y ORDER BY x, y")
with Session(engine) as session:
    result = session.execute(stmt, {"y": 6})
    for row in result:
        print(f"x: {row.x}, y: {row.y}")

2026-01-24 00:05:27,689 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 00:05:27,690 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ? ORDER BY x, y
2026-01-24 00:05:27,691 INFO sqlalchemy.engine.Engine [generated in 0.00058s] (6,)
2026-01-24 00:05:27,692 INFO sqlalchemy.engine.Engine ROLLBACK


In [15]:
# commit #

with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11},{"x": 13, "y": 15}],
    )
    session.commit()

2026-01-24 00:05:30,229 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 00:05:30,230 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-24 00:05:30,231 INFO sqlalchemy.engine.Engine [generated in 0.00050s] ((11, 9), (15, 13))
2026-01-24 00:05:30,231 INFO sqlalchemy.engine.Engine COMMIT


In [16]:
with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11}, {"x": 13, "y": 15}]
    )
    session.commit()

2026-01-24 00:05:33,823 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-24 00:05:33,824 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-24 00:05:33,825 INFO sqlalchemy.engine.Engine [cached since 3.595s ago] ((11, 9), (15, 13))
2026-01-24 00:05:33,827 INFO sqlalchemy.engine.Engine COMMIT


<h2>Setting up MetaData with Table objects<h2/>

In [17]:
from sqlalchemy import MetaData
metadata_obj = MetaData()

In [18]:
from sqlalchemy import Table, Column, Integer, String
user_table = Table(
    "user_account",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("name", String(30)),
    Column("fullname", String),
)

In [26]:
user_table.c.name

Column('name', String(length=30), table=<user_account>)

In [27]:
user_table.c.keys()

['id', 'name', 'fullname']

In [28]:
user_table.primary_key

PrimaryKeyConstraint(Column('id', Integer(), table=<user_account>, primary_key=True, nullable=False))